# NYC Building Energy Benchmarking - Part H: Association Rule Mining
## TY B.Tech Integrated Data Warehouse, OLAP and Data Mining Project
**Domain:** Building Energy and Sustainability Analytics (NYC Local Law 84)

### Objectives:
1. Ingest building performance mart from `data/mart/building_performance_mart.csv`
2. Discretize continuous attributes into categorical transaction items:
   - `PropertyTypeGroup`
   - `SizeBand`
   - `Borough`
   - `EnergyIntensityBand`
   - `GHGIntensityBand`
   - `EnergyStarBand`
3. Generate frequent itemsets via Apriori / FP-Growth concepts
4. Discover association rules meeting Support >= 0.05 and Confidence >= 0.50
5. Rank by Lift and interpret top 10 rules with actionable domain implications


In [1]:
import os
import itertools
import numpy as np
import pandas as pd

MART_PATH = '../data/mart/building_performance_mart.csv'
df = pd.read_csv(MART_PATH)
print(f"Loaded mart: {len(df):,} records")


Loaded mart: 7,370 records


### 1. Itemset Transaction Construction

In [2]:
transactions = []
for _, row in df.iterrows():
    itemset = [
        f"Type={row['PropertyTypeGroup']}",
        f"Size={row['SizeBand']}",
        f"Borough={row['Borough']}",
        f"EnergyBand={row['EnergyIntensityBand']}",
        f"GHGBand={row['GHGIntensityBand']}",
        f"StarBand={row['EnergyStarBand']}"
    ]
    transactions.append(itemset)

print(f"Example Transaction 0: {transactions[0]}")
N = len(transactions)


Example Transaction 0: ['Type=Residential', 'Size=Medium', 'Borough=Brooklyn', 'EnergyBand=Medium', 'GHGBand=Medium', 'StarBand=Excellent']


### 2. Frequent Itemset Generation & Rule Mining Algorithm

In [3]:
item_counts = {}
for t in transactions:
    for item in t:
        item_counts[item] = item_counts.get(item, 0) + 1

min_support = 0.05
min_confidence = 0.50

frequent_1 = {k: v/N for k, v in item_counts.items() if v/N >= min_support}

pair_counts = {}
for t in transactions:
    frequent_in_t = [item for item in t if item in frequent_1]
    for item_a, item_b in itertools.combinations(sorted(frequent_in_t), 2):
        pair = (item_a, item_b)
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

frequent_2 = {k: v/N for k, v in pair_counts.items() if v/N >= min_support}

rules = []
for (item_a, item_b), supp_ab in frequent_2.items():
    conf_a_to_b = supp_ab / frequent_1[item_a]
    lift_a_to_b = conf_a_to_b / frequent_1[item_b]
    if conf_a_to_b >= min_confidence:
        rules.append({
            'Antecedent': item_a,
            'Consequent': item_b,
            'Support': round(float(supp_ab), 4),
            'Confidence': round(float(conf_a_to_b), 4),
            'Lift': round(float(lift_a_to_b), 4)
        })
    
    conf_b_to_a = supp_ab / frequent_1[item_b]
    lift_b_to_a = conf_b_to_a / frequent_1[item_a]
    if conf_b_to_a >= min_confidence:
        rules.append({
            'Antecedent': item_b,
            'Consequent': item_a,
            'Support': round(float(supp_ab), 4),
            'Confidence': round(float(conf_b_to_a), 4),
            'Lift': round(float(lift_b_to_a), 4)
        })

df_rules = pd.DataFrame(rules).sort_values(by='Lift', ascending=False).drop_duplicates()
print(f"Total Rules Identified (min_supp={min_support}, min_conf={min_confidence}): {len(df_rules)}")


Total Rules Identified (min_supp=0.05, min_conf=0.5): 72


### 3. Top 10 Association Rules with Domain Interpretations

In [4]:
top_10_rules = df_rules.head(10).reset_index(drop=True)
top_10_rules


In [5]:
print("=== Top Association Rules Interpretation ===")
for idx, r in top_10_rules.iterrows():
    print(f"Rule #{idx+1}: IF {r['Antecedent']} THEN {r['Consequent']}")
    print(f"  Support = {r['Support']*100:.1f}%, Confidence = {r['Confidence']*100:.1f}%, Lift = {r['Lift']:.2f}")
    if r['Lift'] > 1.2:
        print("  -> Significant positive association (Lift > 1.2): Target for policy/intervention")
    print()


=== Top Association Rules Interpretation ===
Rule #1: IF EnergyBand=Low THEN Type=Other
  Support = 5.5%, Confidence = 60.5%, Lift = 7.23
  -> Significant positive association (Lift > 1.2): Target for policy/intervention

Rule #2: IF Type=Other THEN EnergyBand=Low
  Support = 5.5%, Confidence = 65.3%, Lift = 7.23
  -> Significant positive association (Lift > 1.2): Target for policy/intervention

Rule #3: IF EnergyBand=Low THEN GHGBand=Low
  Support = 9.0%, Confidence = 100.0%, Lift = 4.00
  -> Significant positive association (Lift > 1.2): Target for policy/intervention

Rule #4: IF EnergyBand=High THEN GHGBand=Very High
  Support = 10.4%, Confidence = 84.2%, Lift = 3.37
  -> Significant positive association (Lift > 1.2): Target for policy/intervention

Rule #5: IF Type=Other THEN GHGBand=Low
  Support = 5.7%, Confidence = 68.5%, Lift = 2.74
  -> Significant positive association (Lift > 1.2): Target for policy/intervention

Rule #6: IF StarBand=Fair THEN GHGBand=Very High
  Support = 5